# 30문항 Claude 채점 시도

3개 사업 × 10문항, 기본 BM25와 Claude 채점 시도. 당시 Claude API 오류가 기록된 실패 실험입니다.

실행 출력은 GitHub 공유를 위해 제거했습니다. 원본 문서와 평가 데이터는 별도로 준비해야 합니다.


# RFP RAG: 기본 BM25 · 30문항 · LangSmith

첫 3개 사업에서 **각 10문항, 총 30문항**을 선택합니다. 질문 → 사업 식별 →
BGE-M3 / 기본 BM25 → RRF → 재순위화 → GPT 답변 → Claude 평가 → LangSmith 저장 순서입니다.

1. Colab에서 **새 GPU 런타임**을 선택하고 위에서 아래로 실행하세요.
2. Drive의 `/content/drive/MyDrive/data`에 RFP 문서를,
   `/content/drive/MyDrive/data_list.csv`에 목록표를 두세요.
3. 동봉한 `rag_eval_dataset_3x10.json`을 Drive 루트에 두세요.
   기존 `rag_eval_dataset.json`도 사용할 수 있습니다. 처음 등장한 3개 사업의 첫 10문항씩 선택합니다.
4. Colab 보안 비밀에 `LANGSMITH_API_KEY`, `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`를 등록하고 접근을 허용하세요.
   워크스페이스에 한정되지 않은 Claude 키는 `ANTHROPIC_WORKSPACE_ID`도 등록해야 합니다.
   값은 Claude Console의 Settings → Workspaces → ID에서 복사합니다.
5. 사전 검사에서 첫 문항의 답변 생성과 Claude 채점을 실제로 확인합니다.
   마지막 셀은 그 결과를 재사용하여 나머지 문항을 순차 실행합니다.
6. 성공 여부는 **30개 ID 일치 · 생성 오류 없음 · Claude 채점 30개 · 사업 식별 30개 ·
   LangSmith 원격 실행/피드백 저장 확인**으로 판단합니다. CSV는 오류가 있어도 먼저 저장합니다.

API 속도 제한과 일시적인 연결 오류는 SDK에서 최대 4회 재시도합니다.
같은 세션에서 마지막 셀만 재실행하면 성공한 답변/채점을 메모리 캐시에서 재사용합니다.
런타임을 새로 시작하면 다시 API를 호출합니다. API 이용 요금은 각 제공자의 계정에 청구됩니다.

이 파일은 기본 BM25(`BM25_MODE="current"`)를 사용합니다.
Claude의 `temperature`는 전달하지 않습니다.
참고: [Anthropic SDK 호출 정의](https://github.com/anthropics/anthropic-sdk-python/blob/main/src/anthropic/resources/messages/messages.py),
[워크스페이스 인증](https://platform.claude.com/docs/en/manage-claude/authentication),
[LangSmith 평가 API](https://reference.langchain.com/python/langsmith/evaluation).


In [ ]:
%pip install -q numpy pandas faiss-cpu "sentence-transformers==5.6.0" langchain-huggingface langchain-community langchain-text-splitters pymupdf4llm langchain-hwp-hwpx-loader "transformers==4.51.3" accelerate rank_bm25 kiwipiepy
%pip install -U -q langsmith openai anthropic langchain-openai


In [ ]:
import os
from importlib.metadata import version

def require_secret(name):
    if not os.getenv(name):
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
    if not os.getenv(name):
        raise RuntimeError(f"{name}을 Colab 보안 비밀 또는 환경 변수에 설정하세요.")

for secret_name in ("LANGSMITH_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    require_secret(secret_name)
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "rfp-bm25-3x10"
print("API 키 설정 완료 (키 값은 출력하지 않습니다).")
print({name: version(name) for name in ("langsmith", "openai", "anthropic")})


In [ ]:
import os
import re
import unicodedata

import pandas as pd
import pymupdf4llm
from google.colab import drive
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from hwp_hwpx_parser import ExtractOptions, TableStyle
from langchain_text_splitters import RecursiveCharacterTextSplitter

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/data"
metadata_csv = "/content/drive/MyDrive/data_list.csv"


In [ ]:
# 문서 순서를 고정해 다시 실행해도 청크 번호가 같은 순서로 생성되도록 합니다.
sorted_file_names = sorted(
    os.listdir(folder),
    key=lambda name: (unicodedata.normalize("NFKC", name).casefold(), name),
)
print("정렬한 파일 수:", len(sorted_file_names))

## 1. 문서 로드 (HWP, PDF)


In [ ]:
def normalize_filename(name):
    name = str(name).rsplit(".", 1)[0]
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip()

df = pd.read_csv(metadata_csv, encoding="utf-8-sig")
loaded_files = []
for name in sorted_file_names:
    full_path = os.path.join(folder, name)
    if not os.path.isfile(full_path):
        continue
    extension = name.rsplit(".", 1)[-1].lower()
    if extension == "pdf":
        text = pymupdf4llm.to_markdown(full_path)
    elif extension in ("hwp", "hwpx"):
        options = ExtractOptions(table_style=TableStyle.MARKDOWN)
        loader = HwpHwpxLoader(full_path, mode="single", extract_options=options)
        text = "\n\n".join(doc.page_content for doc in loader.load())
    else:
        continue
    if not text.strip():
        raise ValueError(f"본문이 비어 있습니다: {name}")
    loaded_files.append((name, text))
if not loaded_files:
    raise ValueError(f"PDF/HWP/HWPX 문서를 찾지 못했습니다: {folder}")
print("로드한 문서:", len(loaded_files))


## 2. 메타데이터와 본문을 Document로 저장

`page_content`에 본문을, `metadata`에 CSV의 문서 정보를 저장합니다.


In [ ]:
metadata_columns = [
    "공고 번호", "공고 차수", "사업명", "사업 금액", "발주 기관",
    "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일", "파일명",
]
missing_columns = set(metadata_columns) - set(df.columns)
if missing_columns:
    raise ValueError(f"data_list.csv에 없는 열: {sorted(missing_columns)}")
documents = []
unmatched_files = []
for name, text in loaded_files:
    result_df = df[df["파일명"].apply(normalize_filename) == normalize_filename(name)]
    if result_df.empty:
        unmatched_files.append(name)
        continue
    if len(result_df[["발주 기관", "사업명"]].drop_duplicates()) != 1:
        raise ValueError(f"파일의 사업 메타데이터가 여러 개입니다: {name}")
    row = result_df.iloc[0]
    metadata = {column: "" if pd.isna(row[column]) else str(row[column])
                for column in metadata_columns}
    documents.append(Document(page_content=text, metadata=metadata))
if not documents:
    raise ValueError("문서 파일명과 data_list.csv의 파일명을 확인하세요.")
print("메타데이터와 연결한 문서:", len(documents))
if unmatched_files:
    print("CSV와 연결되지 않아 제외한 파일:", unmatched_files)


## 3. 본문 기본 전처리

연속 공백과 빈 줄만 정리합니다.


In [ ]:
for document in documents:
    text = document.page_content.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    document.page_content = re.sub(r"\n{3,}", "\n\n", text).strip()


## 4. 청킹


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " "], chunk_size=700, chunk_overlap=100
)

chunks = splitter.split_documents(documents)
for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id

print("청크 수:", len(chunks))


## 5. 임베딩 모델 준비

사업별 FAISS 인덱스는 해당 사업의 질문이 처음 들어올 때 만듭니다.


In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

model_device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "dragonkue/BGE-m3-ko"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": model_device, "trust_remote_code": True},
    encode_kwargs={"batch_size": 16, "normalize_embeddings": True},
)
print("임베딩 모델:", embedding_model_name, "/ 장치:", model_device)


## 6. 질문에서 사업 식별 후 하이브리드 검색

질문에 적힌 정식 사업명을 문서 메타데이터의 사업명 목록과 비교합니다. 띄어쓰기와 문장부호 차이는 무시합니다. 줄임말은 `PROJECT_ALIASES`에 등록하세요. 사업을 하나로 특정할 수 없으면 오류를 내며, 평가 데이터의 `project` 필드는 검색에 사용하지 않습니다.


In [ ]:
def project_title_matches_query(project_key, normalized_query):
    agency, title = project_key
    normalized_title = normalize_project_text(title)
    if normalized_title in normalized_query:
        return True

    # 기관명이 사업명 앞에 있지만 질문에서는 조사와 함께 분리된 경우
    normalized_agency = normalize_project_text(agency)
    if normalized_agency and normalized_title.startswith(normalized_agency):
        short_title = normalized_title[len(normalized_agency):]
        if len(short_title) >= 8 and normalized_agency in normalized_query:
            return short_title in normalized_query
    return False


def project_identity_matches(project_key, expected_project):
    agency, title = project_key
    expected = normalize_project_text(expected_project)
    return expected in {
        normalize_project_text(title),
        normalize_project_text(agency + title),
    }

In [ ]:
from collections import defaultdict
from langchain_community.retrievers import BM25Retriever


def normalize_project_text(value):
    return "".join(
        char for char in unicodedata.normalize("NFKC", str(value)).casefold()
        if char.isalnum()
    )


def metadata_text(value):
    return "" if pd.isna(value) else str(value).strip()


# 줄임말을 쓰려면 아래에 "줄임말": ("발주 기관", "정식 사업명")을 추가합니다.
PROJECT_ALIASES = {}

project_chunks_by_key = defaultdict(list)
for doc in chunks:
    agency = metadata_text(doc.metadata.get("발주 기관"))
    title = metadata_text(doc.metadata.get("사업명"))
    if title:
        project_chunks_by_key[(agency, title)].append(doc)

project_catalog = sorted(project_chunks_by_key)
if not project_catalog:
    raise ValueError("사업명 메타데이터가 있는 청크가 없습니다.")
print("식별 가능한 사업:", len(project_catalog))


def choose_project(matches, query):
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0]

    normalized_query = normalize_project_text(query)
    agency_matches = [
        key for key in matches
        if key[0] and normalize_project_text(key[0]) in normalized_query
    ]
    if len(agency_matches) == 1:
        return agency_matches[0]

    labels = ", ".join(f'[{agency}] "{title}"' for agency, title in matches)
    raise ValueError(f"사업을 하나로 특정할 수 없습니다: {labels}")


def identify_project(query):
    normalized_query = normalize_project_text(query)
    if not normalized_query:
        raise ValueError("질문이 비어 있습니다.")

    # 정식 사업명이 질문에 들어 있으면 포함 관계의 짧은 이름만 제외합니다.
    direct_matches = [
      key for key in project_catalog
      if project_title_matches_query(key, normalized_query)
    ]
    if direct_matches:
        specific_matches = [
            key for key in direct_matches
            if not any(
                normalize_project_text(key[1]) != normalize_project_text(other[1])
                and normalize_project_text(key[1]) in normalize_project_text(other[1])
                for other in direct_matches
            )
        ]
        return choose_project(specific_matches, query)

    # 정식 명칭이 없는 질문은 사용자가 등록한 줄임말만 사용합니다.
    alias_matches = [
        (alias, target) for alias, target in PROJECT_ALIASES.items()
        if normalize_project_text(alias) in normalized_query
    ]
    if alias_matches:
        longest = max(len(normalize_project_text(alias)) for alias, _ in alias_matches)
        targets = [
            target for alias, target in alias_matches
            if len(normalize_project_text(alias)) == longest
        ]
        unknown = [target for target in targets if target not in project_chunks_by_key]
        if unknown:
            raise ValueError(f"PROJECT_ALIASES의 사업이 문서 목록에 없습니다: {unknown}")
        return choose_project(targets, query)

    raise ValueError(
        "질문에서 사업명을 찾지 못했습니다. 정식 사업명을 질문에 적거나 "
        "PROJECT_ALIASES에 줄임말을 등록해 주세요."
    )


from kiwipiepy import Kiwi

kiwi = Kiwi()
BM25_MODES = ("current", "kiwi")
BM25_MODE = "current"
PUNCTUATION_TAGS = {"SF", "SP", "SS", "SE", "SO", "SW"}


def kiwi_morphs(text):
    """Tokenize documents and queries identically for Kiwi BM25."""
    return [
        token.form
        for token in kiwi.tokenize(text)
        if token.form.strip() and token.tag not in PUNCTUATION_TAGS
    ]


dense_retrievers_by_project = {}
bm25_retrievers_by_project_and_mode = {}


def get_retrievers(project_key, bm25_mode):
    if bm25_mode not in BM25_MODES:
        raise ValueError(f"지원하지 않는 BM25 방식: {bm25_mode}")

    project_chunks = project_chunks_by_key[project_key]
    if project_key not in dense_retrievers_by_project:
        store = FAISS.from_documents(project_chunks, embeddings)
        dense_retrievers_by_project[project_key] = store.as_retriever(search_kwargs={"k": 20})

    bm25_key = (project_key, bm25_mode)
    if bm25_key not in bm25_retrievers_by_project_and_mode:
        if bm25_mode == "kiwi":
            bm25 = BM25Retriever.from_documents(
                project_chunks, preprocess_func=kiwi_morphs
            )
        else:
            bm25 = BM25Retriever.from_documents(project_chunks)
        bm25.k = 20
        bm25_retrievers_by_project_and_mode[bm25_key] = bm25

    return (
        dense_retrievers_by_project[project_key],
        bm25_retrievers_by_project_and_mode[bm25_key],
    )


def rrf_search(query, project_key, bm25_mode, k=60):
    dense_retriever, bm25_retriever = get_retrievers(project_key, bm25_mode)
    dense_results = dense_retriever.invoke(query)
    sparse_results = bm25_retriever.invoke(query)
    scores = {}

    for results in (dense_results, sparse_results):
        for rank, doc in enumerate(results, start=1):
            doc_id = doc.metadata["chunk_id"]
            if doc_id not in scores:
                scores[doc_id] = {"doc": doc, "score": 0}
            scores[doc_id]["score"] += 1 / (k + rank)

    ranked = sorted(scores.values(), key=lambda item: item["score"], reverse=True)
    return [item["doc"] for item in ranked]


## 7. HyDE로 검색 문장 만들기


In [ ]:
import gc
import torch

"del embeddings"

gc.collect()
torch.cuda.empty_cache()

In [ ]:
hyde_search_rules = '''너는 RFP에서 관련 문단을 찾기 위한 가상 안내문 작성자야.
실제 답변이 아니라 검색에 쓸 안내문 1문장만 작성해.
- 질문의 핵심 주제어를 그대로 포함해.
- 질문과 관계없는 다른 사업이나 항목으로 이동하지 마.
- 확인되지 않은 금액, 날짜, 법률, 자격 조건을 단정하지 마.
- 제목, 목록, 따옴표, 추가 설명을 쓰지 마.

예시:
질문: 이 사업의 입찰 참여 자격은 무엇인가요?
가상 안내문: 이 사업의 입찰 참여 자격에 관한 조건이 안내되어 있다.'''

In [ ]:
# 이 비교에서는 아래 셀의 GPT-5-mini를 사용합니다.
# 기존 EXAONE AWQ 로딩은 실행하지 않습니다.
method = "basic"


In [ ]:
from langchain_openai import ChatOpenAI

method = "basic"  # "basic" 또는 "hyde"
GENERATION_MODEL = "gpt-5-mini-2025-08-07"
chat_model = ChatOpenAI(
    model=GENERATION_MODEL, timeout=180, max_retries=4,
)


In [ ]:
def make_search_text(query, method):
    if method == "basic":
        return query
    if method == "hyde":
        hyde_prompt = f"{hyde_search_rules}\n\n질문: {query}\n가상 안내문:"
        return chat_model.invoke(hyde_prompt).content.strip()
    raise ValueError(f"지원하지 않는 검색 방식: {method}")

## 8. 질문에서 식별한 사업 안에서 재순위화


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "Alibaba-NLP/gte-multilingual-reranker-base",
    trust_remote_code=True,
    device=model_device,
)

# 질문과 검색 설정이 같을 때만 검색 결과를 재사용합니다.
retrieval_cache = {}


def retrieve_and_rerank(query, method, bm25_mode=None):
    bm25_mode = BM25_MODE if bm25_mode is None else bm25_mode
    project_key = identify_project(query)
    cache_key = (project_key, method, query, bm25_mode)
    if cache_key not in retrieval_cache:
        search_text = make_search_text(query, method)
        found = rrf_search(search_text, project_key, bm25_mode)
        ranking = reranker.rank(
            query,
            [doc.page_content for doc in found],
            top_k=6,
            batch_size=8,
        )
        reranked_docs = [found[row["corpus_id"]] for row in ranking]
        retrieval_cache[cache_key] = (
            project_key, search_text, found, reranked_docs
        )
    return retrieval_cache[cache_key]


## 9. 일반 질문 함수

아래 함수 셀까지 실행하면 `answer_question("정식 사업명의 공고기간은?")`처럼 평가셋 밖의 질문에도 사용할 수 있습니다. 정식 명칭을 줄여 부르려면 검색 셀의 `PROJECT_ALIASES`에 등록하세요.


In [ ]:
import gc
import torch

"del embeddings"

gc.collect()
torch.cuda.empty_cache()

In [ ]:
answer_rules = '''
너는 RFP 문서의 사실 확인 담당자다. 아래 <문서>에 적힌 내용만으로 <질문>에 답한다.

규칙:
1. 질문이 여러 항목을 묻는다면 항목별로 근거를 따로 찾는다.
2. 질문의 사업명과 다른 사업의 정보는 사용하지 않는다.
3. 각 항목에 대해 문서에서 직접 인용할 수 있는 문구를 먼저 제시한다.
4. 직접 인용할 근거가 없는 항목은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 쓴다.
5. 금액, 기간, 날짜, 비율, 인력, 제출 조건은 문서의 표현을 그대로 옮긴다. 계산하거나 일반적인 관행으로 빈칸을 채우지 않는다.
6. 답변의 각 사실 바로 뒤에 근거 문서 번호 [1]처럼 붙인다.
7. HyDE 검색 문장과 모델의 사전 지식은 근거로 사용하지 않는다.

출력 형식:
근거:
- 항목명: [문서 번호] "문서에서 그대로 가져온 짧은 문구"
- 근거가 없는 항목명: 확인 불가

답변:
근거가 확인된 내용만 간결하게 답한다. 근거가 없는 항목은 확인할 수 없다고 밝힌다.

<문서>
{context}
</문서>

<질문>
{query}
</질문>
'''


import hashlib

answer_cache = {}


def answer_question(query, search_method=None, bm25_mode=None):
    bm25_mode = BM25_MODE if bm25_mode is None else bm25_mode
    selected_method = method if search_method is None else search_method
    project_key, search_text, found, reranked_docs = retrieve_and_rerank(
        query, selected_method, bm25_mode
    )
    context = "\n\n".join(
        f"[{j}] 메타데이터: {doc.metadata}\n본문: {doc.page_content}"
        for j, doc in enumerate(reranked_docs, start=1)
    )
    prompt = answer_rules.format(context=context, query=query)
    cache_key = hashlib.sha256(
        f"{GENERATION_MODEL}|{selected_method}|{bm25_mode}|{prompt}".encode("utf-8")
    ).hexdigest()
    if cache_key in answer_cache:
        return dict(answer_cache[cache_key], answer_cached=True)
    content = chat_model.invoke(prompt).content
    answer = content.strip() if isinstance(content, str) else "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    ).strip()
    if not answer:
        raise ValueError(f"답변이 비어 있습니다: {query}")
    record = {
        "query": query,
        "answer_cached": False,
        "bm25_mode": bm25_mode,
        "identified_agency": project_key[0],
        "identified_project": project_key[1],
        "search_text": search_text,
        "answer": answer,
        "contexts": [doc.page_content for doc in reranked_docs],
        "chunk_ids": [doc.metadata["chunk_id"] for doc in reranked_docs],
        "rrf_chunk_ids": [doc.metadata["chunk_id"] for doc in found],
    }
    answer_cache[cache_key] = record
    return record


### 평가 질문 불러오기: 3개 사업 × 10문항

파일에 기록된 사업 순서를 유지하고 각 사업의 첫 10문항을 선택합니다.
기준 답변과 사업 정답은 채점에 사용합니다. 검색의 사업 식별에는 질문만 사용합니다.


In [ ]:
import json
from collections import Counter
from pathlib import Path

EVAL_PROJECT_COUNT = 3
EVAL_QUESTIONS_PER_PROJECT = 10
EXPECTED_QUESTION_COUNT = EVAL_PROJECT_COUNT * EVAL_QUESTIONS_PER_PROJECT
EVAL_LABEL = f"{EVAL_PROJECT_COUNT}x{EVAL_QUESTIONS_PER_PROJECT}"
eval_dataset_candidates = [
    Path("/content/drive/MyDrive/rag_eval_dataset_3x10.json"),
    Path("/content/drive/MyDrive/rag_eval_dataset.json"),
    Path("rag_eval_dataset_3x10.json"),
    Path("rag_eval_dataset.json"),
]
eval_dataset_path = next((p for p in eval_dataset_candidates if p.is_file()), None)
if eval_dataset_path is None:
    raise FileNotFoundError("평가 JSON을 Google Drive 루트 또는 노트북 폴더에 두세요.")
with eval_dataset_path.open(encoding="utf-8-sig") as file:
    all_eval_records = json.load(file)
if not isinstance(all_eval_records, list) or not all_eval_records:
    raise ValueError("평가 JSON은 문항 객체들의 비어 있지 않은 목록이어야 합니다.")
if any(not isinstance(r, dict) or not r.get("project") for r in all_eval_records):
    raise ValueError("사업명(project)이 없는 평가 문항이 있습니다.")
available_projects = list(dict.fromkeys(r["project"] for r in all_eval_records))
selected_projects = available_projects[:EVAL_PROJECT_COUNT]
selected_eval_records = []
selected_count = Counter()
for record in all_eval_records:
    project = record["project"]
    if project in selected_projects and selected_count[project] < EVAL_QUESTIONS_PER_PROJECT:
        selected_eval_records.append(record)
        selected_count[project] += 1
if len(selected_projects) != EVAL_PROJECT_COUNT or any(
    selected_count[p] != EVAL_QUESTIONS_PER_PROJECT for p in selected_projects
):
    raise ValueError(f"3개 사업에서 각 10문항을 선택하지 못했습니다: {dict(selected_count)}")
evaluation_data = []
for record in selected_eval_records:
    fields = ("id", "question", "reference", "project")
    if not all(isinstance(record.get(f), str) and record[f].strip() for f in fields):
        raise ValueError(f"필수 평가 필드가 비어 있거나 문자열이 아닙니다: {record.get('id')}")
    evaluation_data.append({
        "id": record["id"], "query": record["question"],
        "reference": record["reference"], "project": record["project"],
        "question_type": record.get("question_type", ""),
        "evidence_quotes": record.get("evidence_quotes", []),
    })
for field in ("id", "query"):
    if len({r[field] for r in evaluation_data}) != EXPECTED_QUESTION_COUNT:
        raise ValueError(f"문항 수가 맞지 않거나 {field}가 중복됩니다.")
question = evaluation_data
print(f"평가셋: {eval_dataset_path} / {len(evaluation_data)}문항")
print("사업별:", dict(Counter(r["project"] for r in evaluation_data)))
print("유형별:", dict(Counter(r["question_type"] for r in evaluation_data)))


### LangSmith 데이터셋: 30개 예제를 확정한 뒤 실행

문항의 전체 내용으로 데이터셋 이름을 만듭니다. 업로드가 중간에 멈췄던 데이터셋에는
없는 문항만 추가합니다. 저장된 예제 30개의 입력/기준 답변을 검증하고 문항 순서대로 목록을 만듭니다.


In [ ]:
import hashlib
import time
from langsmith import Client

dataset_payload = json.dumps(evaluation_data, ensure_ascii=False, sort_keys=True)
dataset_hash = hashlib.sha256(dataset_payload.encode("utf-8")).hexdigest()[:12]
DATASET_NAME = f"rfp-bm25-{EVAL_LABEL}-{dataset_hash}"
ls_client = Client()
if ls_client.has_dataset(dataset_name=DATASET_NAME):
    dataset = ls_client.read_dataset(dataset_name=DATASET_NAME)
else:
    dataset = ls_client.create_dataset(
        dataset_name=DATASET_NAME,
        description="첫 3개 사업의 각 10문항: 기본 BM25 RAG / Claude 답변 평가",
    )
expected_examples = {r["id"]: {
    "inputs": {"id": r["id"], "question": r["query"], "project": r["project"],
               "question_type": r["question_type"]},
    "outputs": {"reference": r["reference"], "evidence_quotes": r["evidence_quotes"]},
} for r in evaluation_data}

def checked_examples(examples):
    by_id = {}
    for example in examples:
        item_id = example.inputs.get("id")
        if item_id not in expected_examples or item_id in by_id:
            raise RuntimeError(f"LangSmith 데이터셋에 알 수 없거나 중복된 ID: {item_id}")
        expected = expected_examples[item_id]
        if example.inputs != expected["inputs"] or example.outputs != expected["outputs"]:
            raise RuntimeError(f"LangSmith 데이터셋 내용이 다릅니다: {item_id}")
        by_id[item_id] = example
    return by_id

existing_by_id = checked_examples(list(ls_client.list_examples(dataset_id=dataset.id)))
missing_examples = [item for item_id, item in expected_examples.items() if item_id not in existing_by_id]
if missing_examples:
    ls_client.create_examples(dataset_id=dataset.id, examples=missing_examples)
for attempt in range(4):
    existing_by_id = checked_examples(list(ls_client.list_examples(dataset_id=dataset.id)))
    if set(existing_by_id) == set(expected_examples):
        break
    if attempt < 3:
        time.sleep(2)
else:
    raise RuntimeError(f"LangSmith 예제 업로드 누락: {sorted(set(expected_examples) - set(existing_by_id))}")
langsmith_examples = [existing_by_id[r["id"]] for r in evaluation_data]
print("LangSmith 데이터셋:", DATASET_NAME, f"/ {len(langsmith_examples)}문항")


### LangSmith 평가

OpenAI로 답변을 생성하고 Claude가 기준 답변 대비 0~1로 채점합니다. 근거 인용 구절의 정확 일치 포함률과 사업 식별도 함께 기록합니다. Claude 점수는 RAGAS 점수와 다른 척도입니다.


In [ ]:
import json
import re
import unicodedata
import anthropic

CLAUDE_JUDGE_MODEL = os.getenv("CLAUDE_JUDGE_MODEL", "claude-sonnet-4-6")
def optional_colab_secret(name):
    value = os.getenv(name, "").strip()
    if value:
        return value
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""


anthropic_workspace_id = optional_colab_secret("ANTHROPIC_WORKSPACE_ID")
claude_client = anthropic.Anthropic(
    timeout=120,
    max_retries=4,
    default_headers={"anthropic-workspace-id": anthropic_workspace_id} if anthropic_workspace_id else {},
)
print("Claude 워크스페이스 헤더:", "설정됨" if anthropic_workspace_id else "없음 (워크스페이스 한정 키만 사용 가능)")
judge_cache = {}


def parse_judge_response(judge_text):
    # 코드 펜스/앞뒤 설명이 있어도 JSON 객체를 찾아 점수의 형식과 범위를 확인합니다.
    decoder = json.JSONDecoder()
    for position, char in enumerate(judge_text):
        if char != "{":
            continue
        try:
            graded, _ = decoder.raw_decode(judge_text[position:])
        except json.JSONDecodeError:
            continue
        if not isinstance(graded, dict) or "score" not in graded:
            continue
        raw_score = graded["score"]
        if isinstance(raw_score, bool) or not isinstance(raw_score, (int, float)):
            raise ValueError("Claude score는 숫자여야 합니다.")
        score = float(raw_score)
        if not 0 <= score <= 1:
            raise ValueError(f"Claude 점수가 0~1 범위를 벗어났습니다: {score}")
        reason = graded.get("reason")
        if not isinstance(reason, str) or not reason.strip():
            raise ValueError("Claude reason이 비어 있습니다.")
        return {"key": "claude_answer_score", "score": score, "comment": reason.strip()}
    raise ValueError("Claude 응답에서 유효한 평가 JSON을 찾지 못했습니다.")

def langsmith_target(inputs: dict) -> dict:
    identified_key = identify_project(inputs["question"])
    if not project_identity_matches(identified_key, inputs["project"]):
        raise ValueError(
            f"사업 식별 오류: {inputs['id']} / "
            f"식별 결과={identified_key!r} / 평가셋={inputs['project']!r}"
        )

    record = answer_question(inputs["question"], bm25_mode=BM25_MODE)
    answer_text = record["answer"].split("답변:", 1)[-1].strip()
    if not answer_text:
        raise ValueError(f"최종 답변이 비어 있습니다: {inputs['id']}")
    return {
        "answer": answer_text,
        "full_answer": record["answer"],
        "contexts": record["contexts"],
        "chunk_ids": record["chunk_ids"],
        "rrf_chunk_ids": record["rrf_chunk_ids"],
        "identified_project": identified_key[1],
        "identified_agency": identified_key[0],
        "bm25_mode": BM25_MODE,
        "answer_cached": record["answer_cached"],
    }

def claude_answer_score(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    # claude_answer_score 시작 부분
    if not outputs or "answer" not in outputs:
      return {"key": "claude_answer_score", "score": None, "comment": "답변 생성 실패"}
    prompt = (
        "질문, 기준 답변, 생성 답변을 비교해 사실 정확성과 필수 항목의 충족 정도를 0~1로 평가하세요. "
        "추가 설명이 사실과 충돌하면 감점하세요. 기준 답변에 없는 사소한 표현 차이는 감점하지 마세요. "
        "반드시 JSON 객체 하나만 반환하세요: {\"score\": 0.0, \"reason\": \"한 문장 근거\"}.\n\n"
        f"질문: {inputs['question']}\n\n기준 답변: {reference_outputs['reference']}\n\n생성 답변: {outputs['answer']}"
    )
    cache_key = hashlib.sha256(f"{CLAUDE_JUDGE_MODEL}|{prompt}".encode("utf-8")).hexdigest()
    if cache_key in judge_cache:
        return dict(judge_cache[cache_key])
    for attempt in range(3):
        # 최신 SDK에서 받지 않는 temperature 인자를 전달하지 않습니다.
        try:
            message = claude_client.messages.create(
                model=CLAUDE_JUDGE_MODEL,
                max_tokens=1024,
                messages=[{"role": "user", "content": prompt}],
            )
        except anthropic.BadRequestError as exc:
            if "anthropic-workspace-id" in str(exc).lower():
                raise RuntimeError(
                    "Claude 요청에 워크스페이스 ID가 필요합니다. "
                    "Claude Console의 Settings → Workspaces → ID 값을 "
                    "Colab 보안 비밀 ANTHROPIC_WORKSPACE_ID에 저장하고 접근을 허용한 뒤, "
                    "Claude 평가 함수 셀과 사업 식별/API 사전 검사 셀을 다시 실행하세요. "
                    "워크스페이스에 한정된 API 키를 사용하는 방법도 있습니다."
                ) from None
            raise
        judge_text = "".join(block.text for block in message.content if block.type == "text")
        try:
            result = parse_judge_response(judge_text)
        except ValueError as exc:
            if attempt == 2:
                raise ValueError(f"Claude 평가 응답 검증 3회 실패: {inputs['id']} / {exc}") from exc
            prompt += "\n다시 확인: score는 0~1 사이 숫자, reason은 비어 있지 않은 문자열인 JSON만 반환하세요."
            continue
        judge_cache[cache_key] = result
        return dict(result)


def compact(text):
    return re.sub(r"\s+", "", unicodedata.normalize("NFC", str(text)))

def evidence_quote_coverage(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    # evidence_quote_coverage 시작 부분
    if not outputs or "contexts" not in outputs:
      return {"key": "exact_quote_coverage", "score": None, "comment": "답변 생성 실패"}
    quotes = []
    for evidence in reference_outputs.get("evidence_quotes", []) or []:
        for curved, straight in re.findall(r'“([^”]+)”|"([^"]+)"', str(evidence)):
            quote = curved or straight
            if "…" in quote or "..." in quote:
                continue
            normalized = compact(quote)
            if len(normalized) >= 8 and re.search(r"[가-힣A-Za-z]", normalized):
                if normalized not in quotes:
                    quotes.append(normalized)
    contexts = [compact(c) for c in outputs["contexts"]]
    found = sum(any(q in c for c in contexts) for q in quotes)
    return {
        "key": "exact_quote_coverage",
        "score": found / len(quotes) if quotes else None,
        "comment": f"발견 {found}/{len(quotes)}; 인용 구절 0개이면 점수 없음",
    }

def correct_project(inputs: dict, outputs: dict) -> dict:
    if not outputs or "identified_project" not in outputs or "identified_agency" not in outputs:
        return {"key": "correct_project", "score": None, "comment": "답변 생성 실패"}
    identified_key = (outputs["identified_agency"], outputs["identified_project"])
    return {
        "key": "correct_project",
        "score": project_identity_matches(identified_key, inputs["project"]),
    }


In [ ]:
project_check_errors = []
for item in evaluation_data:
    try:
        identified_key = identify_project(item["query"])
        if not project_identity_matches(identified_key, item["project"]):
            project_check_errors.append(
                f"{item['id']}: {identified_key!r} != {item['project']!r}"
            )
    except ValueError as exc:
        project_check_errors.append(f"{item['id']}: {exc}")

if project_check_errors:
    raise ValueError(
        "평가 전 사업 식별 검사 실패:\n" + "\n".join(project_check_errors)
    )
print(f"사업 식별 사전 검사: {len(evaluation_data)}/{EXPECTED_QUESTION_COUNT} 통과")
# 사업별 검색기를 먼저 준비해 GPU 색인 작업을 순차적으로 끝냅니다.
for project_key in dict.fromkeys(identify_project(item["query"]) for item in evaluation_data):
    get_retrievers(project_key, BM25_MODE)
    print("검색기 준비:", project_key[1])

# 첫 문항을 실제로 생성/채점합니다. 마지막 셀에서는 성공한 결과를 캐시에서 재사용합니다.
pilot = langsmith_examples[0]
pilot_output = langsmith_target(pilot.inputs)
pilot_score = claude_answer_score(pilot.inputs, pilot_output, pilot.outputs)
print("첫 문항 API 사전 검사 통과:", pilot.inputs["id"], "/ Claude 점수:", pilot_score["score"])

In [ ]:
# 확정한 예제 목록을 사용하고 순차 실행합니다. 런타임에서 결과를 모두 기다립니다.
results = ls_client.evaluate(
    langsmith_target,
    data=langsmith_examples,
    evaluators=[claude_answer_score, evidence_quote_coverage, correct_project],
    experiment_prefix=f"rfp-{EVAL_LABEL}-{BM25_MODE}-claude",
    description="3개 사업 × 각 10문항, 기본 BM25 RAG 답변과 Claude 평가",
    metadata={
        "question_count": EXPECTED_QUESTION_COUNT,
        "bm25_mode": BM25_MODE, "search_method": method,
        "embedding_model": embedding_model_name,
        "generation_model": GENERATION_MODEL, "judge_model": CLAUDE_JUDGE_MODEL,
        "retrieval_top_k": 20, "final_context_count": 6, "dataset_hash": dataset_hash,
    },
    max_concurrency=0,
    blocking=True,
    error_handling="log",
)
results.wait()
print("LangSmith 실험:", results.url)
print("실험 이름:", results.experiment_name)
result_frame = results.to_pandas()
# 실패 진단에도 생성 답변/오류를 확인할 수 있도록 전체 결과부터 저장합니다.
csv_prefix = f"langsmith_{BM25_MODE}_{EVAL_LABEL}_{dataset_hash}"
full_path = eval_dataset_path.with_name(f"{csv_prefix}_full_results.csv")
result_frame.to_csv(full_path, index=False, encoding="utf-8-sig")
print("전체 결과 CSV:", full_path)

def validate_result_frame(frame):
    expected_ids = {r["id"] for r in evaluation_data}
    actual_ids = frame.get("inputs.id", pd.Series(dtype="str"))
    missing_ids = sorted(expected_ids - set(actual_ids))
    unexpected_ids = sorted(set(actual_ids) - expected_ids)
    if len(frame) != EXPECTED_QUESTION_COUNT or actual_ids.duplicated().any() or missing_ids or unexpected_ids:
        raise RuntimeError(
            f"평가 결과 {len(frame)}/{EXPECTED_QUESTION_COUNT}행 / "
            f"누락 ID={missing_ids}, 예상 밖 ID={unexpected_ids}, 중복={actual_ids.duplicated().any()}"
        )
    errors = frame.get("error", pd.Series(None, index=frame.index, dtype="object"))
    if errors.notna().any():
        failed = frame.loc[errors.notna(), ["inputs.id", "error"]]
        raise RuntimeError("답변 생성 실패 문항:\n" + failed.to_string(index=False))
    for column in ("feedback.claude_answer_score", "feedback.correct_project"):
        if column not in frame:
            raise RuntimeError(f"평가 지표 전체 누락: {column}. LangSmith 실험과 오류 로그를 확인하세요.")
        missing = frame[column].isna()
        if missing.any():
            raise RuntimeError(f"{column} 누락 ID: {frame.loc[missing, 'inputs.id'].tolist()}")
    if not frame["feedback.claude_answer_score"].between(0, 1).all():
        raise RuntimeError("Claude 점수가 0~1 범위를 벗어났습니다.")
    if not frame["feedback.correct_project"].eq(True).all():
        raise RuntimeError("사업 식별에 실패한 문항이 있습니다.")
    if "feedback.exact_quote_coverage" not in frame:
        raise RuntimeError("근거 포함률 평가가 실행되지 않았습니다.")

validate_result_frame(result_frame)

# 로컬 결과 외에 LangSmith에 실행/채점이 실제로 저장되었는지도 확인합니다.
ls_client.flush()
expected_run_ids = {str(run_id) for run_id in result_frame["id"]}
expected_example_ids = {str(example.id) for example in langsmith_examples}
for attempt in range(6):
    remote_runs = list(ls_client.list_runs(project_name=results.experiment_name, is_root=True))
    remote_run_ids = {str(run.id) for run in remote_runs}
    remote_example_ids = {str(run.reference_example_id) for run in remote_runs}
    remote_feedback = list(ls_client.list_feedback(run_ids=list(expected_run_ids)))
    claude_saved = {str(f.run_id) for f in remote_feedback
                    if f.key == "claude_answer_score" and f.score is not None and 0 <= f.score <= 1}
    project_saved = {str(f.run_id) for f in remote_feedback
                     if f.key == "correct_project" and f.score == 1}
    quote_saved = {str(f.run_id) for f in remote_feedback if f.key == "exact_quote_coverage"}
    stored_ok = (
        len(remote_runs) == EXPECTED_QUESTION_COUNT and remote_run_ids == expected_run_ids
        and remote_example_ids == expected_example_ids and all(not run.error for run in remote_runs)
        and claude_saved == expected_run_ids and project_saved == expected_run_ids
        and quote_saved == expected_run_ids
    )
    if stored_ok:
        break
    if attempt < 5:
        print(f"LangSmith 저장 반영 대기 ({attempt + 1}/5): 실행 {len(remote_runs)}개, Claude 채점 {len(claude_saved)}개")
        time.sleep(2 * (attempt + 1))
else:
    raise RuntimeError(
        f"로컬 평가 검사는 통과했지만 LangSmith 저장 확인이 끝나지 않았습니다. "
        f"실행 {len(remote_runs)}/{EXPECTED_QUESTION_COUNT}, Claude 채점 {len(claude_saved)}/{EXPECTED_QUESTION_COUNT}. "
        f"실험 URL을 확인하세요: {results.url}"
    )

score_path = eval_dataset_path.with_name(f"{csv_prefix}_scores.csv")
score_columns = [
    "inputs.id", "inputs.project", "feedback.claude_answer_score",
    "feedback.exact_quote_coverage", "feedback.correct_project",
]
result_frame[score_columns].to_csv(score_path, index=False, encoding="utf-8-sig")
print(f"완료: 답변 {EXPECTED_QUESTION_COUNT}개 / Claude 채점 {EXPECTED_QUESTION_COUNT}개 / LangSmith 저장 확인")
print("Claude 평균 점수:", result_frame["feedback.claude_answer_score"].mean())
print("근거 포함률 평균 (검사 가능한 인용 구절이 있는 문항):", result_frame["feedback.exact_quote_coverage"].mean())
print("요약 CSV:", score_path)
display(result_frame[score_columns])
